# Python practice 12: data engineering patterns

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-practice-12-data-engineering-patterns.ipynb)

13 exercises on configuration, validation, incremental loads, deduplication, change detection, pagination, SCD2, drift, reconciliation and backfills. Each exercise names the [reference entries](../learning/python-reference-12-data-engineering-patterns.ipynb) it practises.

Write each function in the empty cell under its exercise and compare your results with the sample calls. Open the solution cell when you are done or stuck. 4 easy, 8 medium, 1 hard.

| # | exercise | difficulty | practises |
| --- | --- | --- | --- |
| 1 | [Settings from environment variables](#exercise-1-settings-from-environment-variables) | Medium | [Typed settings from the environment](../learning/python-reference-12-data-engineering-patterns.ipynb#typed-settings-from-the-environment) |
| 2 | [Validate and quarantine](#exercise-2-validate-and-quarantine) | Medium | [Validate records and quarantine rejects](../learning/python-reference-12-data-engineering-patterns.ipynb#validate-records-and-quarantine-rejects) |
| 3 | [Keep the latest version per key](#exercise-3-keep-the-latest-version-per-key) | Easy | [Deduplicate: keep the latest version](../learning/python-reference-12-data-engineering-patterns.ipynb#deduplicate-keep-the-latest-version) |
| 4 | [Incremental batch with lookback](#exercise-4-incremental-batch-with-lookback) | Medium | [Watermarks and late data](../learning/python-reference-12-data-engineering-patterns.ipynb#watermarks-and-late-data) |
| 5 | [Backoff with full jitter](#exercise-5-backoff-with-full-jitter) | Easy | [Retries with exponential backoff and jitter](../learning/python-reference-12-data-engineering-patterns.ipynb#retries-with-exponential-backoff-and-jitter) |
| 6 | [Detect changed records](#exercise-6-detect-changed-records) | Medium | [Change detection with content hashes](../learning/python-reference-12-data-engineering-patterns.ipynb#change-detection-with-content-hashes) |
| 7 | [Generator over a paginated API](#exercise-7-generator-over-a-paginated-api) | Medium | [Paginating an API with a cursor](../learning/python-reference-12-data-engineering-patterns.ipynb#paginating-an-api-with-a-cursor) |
| 8 | [SCD type 2 merge](#exercise-8-scd-type-2-merge) | Hard | [Slowly changing dimensions (type 2)](../learning/python-reference-12-data-engineering-patterns.ipynb#slowly-changing-dimensions-type-2) |
| 9 | [Schema drift report](#exercise-9-schema-drift-report) | Medium | [Detect schema drift](../learning/python-reference-12-data-engineering-patterns.ipynb#detect-schema-drift) |
| 10 | [Reconcile source and target counts](#exercise-10-reconcile-source-and-target-counts) | Easy | [Backfills and reconciliation](../learning/python-reference-12-data-engineering-patterns.ipynb#backfills-and-reconciliation) |
| 11 | [Plan a resumable backfill](#exercise-11-plan-a-resumable-backfill) | Medium | [Backfills and reconciliation](../learning/python-reference-12-data-engineering-patterns.ipynb#backfills-and-reconciliation) |
| 12 | [Pseudonymise identifiers](#exercise-12-pseudonymise-identifiers) | Easy | [Pseudonymising personal data](../learning/python-reference-12-data-engineering-patterns.ipynb#pseudonymising-personal-data) |
| 13 | [Batch runner with a summary](#exercise-13-batch-runner-with-a-summary) | Medium | [Batches with bounded memory and a run summary](../learning/python-reference-12-data-engineering-patterns.ipynb#batches-with-bounded-memory-and-a-run-summary) |

*Last reviewed: November 2026, against Python 3.13.*

In [ ]:
# Imports used by the sample calls and the solutions.
import io
from collections import Counter, OrderedDict, defaultdict, deque, namedtuple
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from fractions import Fraction


import random


def caught(fn):
    """Call fn and return the exception it raises, or None."""
    try:
        fn()
    except BaseException as error:  # noqa: BLE001
        return error
    return None

## Exercise 1: Settings from environment variables

**Difficulty:** Medium · about 10 min · **Practises:** [Typed settings from the environment](../learning/python-reference-12-data-engineering-patterns.ipynb#typed-settings-from-the-environment)

*Every job starts by parsing its configuration; do it once and fail with all the problems.*

Write `load_settings(env)` reading `JOB_BUCKET` (required str), `JOB_BATCH_SIZE` (int, default 1000) and `JOB_DRY_RUN` (bool from true/false/1/0/yes/no, any case; default False). Return a dict with keys `bucket`, `batch_size`, `dry_run`. If anything is missing or invalid, raise `ValueError` whose message lists every problem, joined with `; `, in that field order.

**Sample calls**

| call | expected |
| --- | --- |
| `load_settings({"JOB_BUCKET": "s3://raw"})` | `{'bucket': 's3://raw', 'batch_size': 1000, 'dry_run': False}` |
| `load_settings({"JOB_BUCKET": "b", "JOB_BATCH_SIZE": "50", "JOB_DRY_RUN": "YES"})` | `{'bucket': 'b', 'batch_size': 50, 'dry_run': True}` |
| `raises(lambda: load_settings({}), ValueError)` | `True` |

In [ ]:
def load_settings(env):
    # your code here
    pass

In [ ]:
def load_settings(env):
    problems, out = [], {}
    if env.get("JOB_BUCKET"):
        out["bucket"] = env["JOB_BUCKET"]
    else:
        problems.append("JOB_BUCKET is required")
    try:
        out["batch_size"] = int(env.get("JOB_BATCH_SIZE", "1000"))
    except ValueError:
        problems.append("JOB_BATCH_SIZE must be an integer")
    raw = env.get("JOB_DRY_RUN", "false").strip().lower()
    if raw in {"true", "1", "yes"}:
        out["dry_run"] = True
    elif raw in {"false", "0", "no"}:
        out["dry_run"] = False
    else:
        problems.append("JOB_DRY_RUN must be a boolean")
    if problems:
        raise ValueError("; ".join(problems))
    return out

## Exercise 2: Validate and quarantine

**Difficulty:** Medium · about 10 min · **Practises:** [Validate records and quarantine rejects](../learning/python-reference-12-data-engineering-patterns.ipynb#validate-records-and-quarantine-rejects)

*Good rows load; bad rows go to quarantine with every reason, so they can be fixed and replayed.*

Write `split_rows(rows)` returning `(good, rejects)`. A row is rejected when `id` is missing, `amount` is not a non-negative int/float (bools are not numbers), or `currency` is not one of USD, EUR, GBP. Each reject is `{"id": row.get("id"), "reasons": [...]}` with reasons `missing id`, `bad amount`, `bad currency` in that order.

**Sample calls**

| call | expected |
| --- | --- |
| `split_rows([{"id": 1, "amount": 5, "currency": "USD"}, {"amount": -1, "currency": "JPY"}])` | `([{'id': 1, 'amount': 5, 'currency': 'USD'}], [{'id': None, 'reasons': ['missing id', 'bad amount', 'bad currency']}])` |
| `split_rows([])` | `([], [])` |
| `split_rows([{"id": 2, "amount": True, "currency": "EUR"}])[1]` | `[{'id': 2, 'reasons': ['bad amount']}]` |

In [ ]:
def split_rows(rows):
    # your code here
    pass

In [ ]:
def split_rows(rows):
    good, rejects = [], []
    for row in rows:
        reasons = []
        if row.get("id") is None:
            reasons.append("missing id")
        amount = row.get("amount")
        if isinstance(amount, bool) or not isinstance(amount, (int, float)) or amount < 0:
            reasons.append("bad amount")
        if row.get("currency") not in {"USD", "EUR", "GBP"}:
            reasons.append("bad currency")
        if reasons:
            rejects.append({"id": row.get("id"), "reasons": reasons})
        else:
            good.append(row)
    return good, rejects

## Exercise 3: Keep the latest version per key

**Difficulty:** Easy · about 5 min · **Practises:** [Deduplicate: keep the latest version](../learning/python-reference-12-data-engineering-patterns.ipynb#deduplicate-keep-the-latest-version)

*CDC feeds and at-least-once queues deliver several versions of a record.*

Write `latest(rows)` keeping, per `id`, the row with the greatest `(updated_at, seq)` and returning the kept rows' `(id, status)` sorted by id.

**Sample calls**

| call | expected |
| --- | --- |
| `latest([{"id": 1, "status": "a", "updated_at": 1, "seq": 1}, {"id": 1, "status": "b", "updated_at": 2, "seq": 2}])` | `[(1, 'b')]` |
| `latest([{"id": 1, "status": "late-seq", "updated_at": 5, "seq": 9}, {"id": 1, "status": "early-seq", "updated_at": 5, "seq": 3}])` | `[(1, 'late-seq')]` |
| `latest([])` | `[]` |

In [ ]:
def latest(rows):
    # your code here
    pass

In [ ]:
def latest(rows):
    best = {}
    for r in rows:
        if r["id"] not in best or (r["updated_at"], r["seq"]) > (best[r["id"]]["updated_at"], best[r["id"]]["seq"]):
            best[r["id"]] = r
    return [(k, best[k]["status"]) for k in sorted(best)]

## Exercise 4: Incremental batch with lookback

**Difficulty:** Medium · about 10 min · **Practises:** [Watermarks and late data](../learning/python-reference-12-data-engineering-patterns.ipynb#watermarks-and-late-data)

*Incremental loads must catch late-arriving records without moving the watermark backwards.*

Write `next_batch(rows, watermark, lookback_minutes=60)` with integer minute timestamps in `ts`; return `(ids, new_watermark)` where ids are the sorted ids of rows with `ts > watermark - lookback_minutes`, and new_watermark is the max of the old watermark and the batch's max `ts`.

**Sample calls**

| call | expected |
| --- | --- |
| `next_batch([{"id": 1, "ts": 100}, {"id": 2, "ts": 990}, {"id": 3, "ts": 1010}], 1000)` | `([2, 3], 1010)` |
| `next_batch([{"id": 1, "ts": 5}], 1000)` | `([], 1000)` |
| `next_batch([{"id": 1, "ts": 950}], 1000)[1]` | `1000` |

In [ ]:
def next_batch(rows, watermark, lookback_minutes=60):
    # your code here
    pass

In [ ]:
def next_batch(rows, watermark, lookback_minutes=60):
    batch = [r for r in rows if r["ts"] > watermark - lookback_minutes]
    return sorted(r["id"] for r in batch), max([watermark] + [r["ts"] for r in batch])

## Exercise 5: Backoff with full jitter

**Difficulty:** Easy · about 5 min · **Practises:** [Retries with exponential backoff and jitter](../learning/python-reference-12-data-engineering-patterns.ipynb#retries-with-exponential-backoff-and-jitter)

*Retry delays must grow, be capped, and be randomised to avoid synchronised retries.*

Write `delays(attempts, base=1.0, cap=20.0, rng=random.random)` returning, for attempt i (0-based), `rng() * min(cap, base * 2**i)` rounded to 3 decimals. Checks inject `rng`.

**Sample calls**

| call | expected |
| --- | --- |
| `delays(6, rng=lambda: 1.0)` | `[1.0, 2.0, 4.0, 8.0, 16.0, 20.0]` |
| `delays(4, base=0.5, rng=lambda: 0.5)` | `[0.25, 0.5, 1.0, 2.0]` |
| `delays(0)` | `[]` |

In [ ]:
def delays(attempts, base=1.0, cap=20.0, rng=None):
    # your code here
    pass

In [ ]:
import random

def delays(attempts, base=1.0, cap=20.0, rng=random.random):
    return [round(rng() * min(cap, base * 2 ** i), 3) for i in range(attempts)]

## Exercise 6: Detect changed records

**Difficulty:** Medium · about 10 min · **Practises:** [Change detection with content hashes](../learning/python-reference-12-data-engineering-patterns.ipynb#change-detection-with-content-hashes)

*Snapshot diffs tell you what to upsert and delete.*

Write `snapshot_diff(old, new, ignore=("loaded_at",))` returning `{"inserted": [...], "updated": [...], "deleted": [...]}` of sorted ids, comparing SHA-256 hashes of each record serialised as JSON with sorted keys and the ignored fields removed.

**Sample calls**

| call | expected |
| --- | --- |
| `snapshot_diff([{"id": 1, "v": 1}, {"id": 2, "v": 1}], [{"id": 2, "v": 2}, {"id": 3, "v": 1}])` | `{'inserted': [3], 'updated': [2], 'deleted': [1]}` |
| `snapshot_diff([{"id": 1, "v": 1, "loaded_at": "mon"}], [{"id": 1, "v": 1, "loaded_at": "tue"}])` | `{'inserted': [], 'updated': [], 'deleted': []}` |
| `snapshot_diff([{"id": 1, "a": 1, "b": 2}], [{"b": 2, "id": 1, "a": 1}])` | `{'inserted': [], 'updated': [], 'deleted': []}` |

In [ ]:
def snapshot_diff(old, new, ignore=("loaded_at",)):
    # your code here
    pass

In [ ]:
import hashlib
import json

def _hash(record, ignore):
    canonical = json.dumps({k: v for k, v in record.items() if k not in ignore}, sort_keys=True, default=str)
    return hashlib.sha256(canonical.encode()).hexdigest()

def snapshot_diff(old, new, ignore=("loaded_at",)):
    a = {r["id"]: _hash(r, ignore) for r in old}
    b = {r["id"]: _hash(r, ignore) for r in new}
    return {"inserted": sorted(b.keys() - a.keys()), "updated": sorted(k for k in a.keys() & b.keys() if a[k] != b[k]), "deleted": sorted(a.keys() - b.keys())}

## Exercise 7: Generator over a paginated API

**Difficulty:** Medium · about 10 min · **Practises:** [Paginating an API with a cursor](../learning/python-reference-12-data-engineering-patterns.ipynb#paginating-an-api-with-a-cursor)

*Hide paging behind a generator and protect against broken cursors.*

Write a generator `iter_items(fetch_page, max_pages=100)` calling `fetch_page(cursor)` (starting with None) which returns `{"items": [...], "next": cursor_or_None}`, yielding items until `next` is None. Raise `RuntimeError` if a cursor repeats or `max_pages` is exceeded. Checks use `list(...)`.

**Sample calls**

| call | expected |
| --- | --- |
| `list(iter_items({None: {"items": [1, 2], "next": "b"}, "b": {"items": [3], "next": "c"}, "c": {"items": [], "next": None}}.__getitem__))` | `[1, 2, 3]` |
| `list(iter_items(lambda c: {"items": ["x"], "next": None}))` | `['x']` |
| `list(iter_items(lambda c: {"items": [], "next": None}))` | `[]` |

In [ ]:
def iter_items(fetch_page, max_pages=100):
    # your code here
    pass

In [ ]:
def iter_items(fetch_page, max_pages=100):
    cursor, seen = None, set()
    for _ in range(max_pages):
        page = fetch_page(cursor)
        yield from page["items"]
        cursor = page.get("next")
        if cursor is None:
            return
        if cursor in seen:
            raise RuntimeError(f"cursor {cursor!r} repeated")
        seen.add(cursor)
    raise RuntimeError("too many pages")

## Exercise 8: SCD type 2 merge

**Difficulty:** Hard · about 20 min · **Practises:** [Slowly changing dimensions (type 2)](../learning/python-reference-12-data-engineering-patterns.ipynb#slowly-changing-dimensions-type-2)

*History-preserving dimension updates are a staple data-modelling interview question.*

Write `scd2_merge(dim, updates, as_of)` where dim rows are `{id, email, valid_from, valid_to, is_current}` and updates are `{id, email}`. For a changed email, close the current row (`valid_to=as_of`, `is_current=False`) and add a new current row from `as_of`; for a new id add a current row; unchanged ids are untouched. Do not modify the input; return rows sorted by `(id, valid_from)`.

**Sample calls**

| call | expected |
| --- | --- |
| `scd2_merge([{"id": 1, "email": "a", "valid_from": "d1", "valid_to": None, "is_current": True}], [{"id": 1, "email": "b"}, {"id": 2, "email": "c"}], "d2")` | `[{'id': 1, 'email': 'a', 'valid_from': 'd1', 'valid_to': 'd2', 'is_current': False}, {'id': 1, 'email': 'b', 'valid_from': 'd2', 'valid_to': None, 'is_current': True}, {'id': 2, 'email': 'c', 'valid_from': 'd2', 'valid_to': None, 'is_current': True}]` |
| `scd2_merge([{"id": 1, "email": "a", "valid_from": "d1", "valid_to": None, "is_current": True}], [{"id": 1, "email": "a"}], "d2")` | `[{'id': 1, 'email': 'a', 'valid_from': 'd1', 'valid_to': None, 'is_current': True}]` |
| `len(scd2_merge([{"id": 1, "email": "a", "valid_from": "d0", "valid_to": "d1", "is_current": False}, {"id": 1, "email": "b", "valid_from": "d1", "valid_to": None, "is_current": True}], [{"id": 1, "email": "c"}], "d2"))` | `3` |

In [ ]:
def scd2_merge(dim, updates, as_of):
    # your code here
    pass

In [ ]:
def scd2_merge(dim, updates, as_of):
    out = [dict(r) for r in dim]
    current = {r["id"]: r for r in out if r["is_current"]}
    for u in updates:
        row = current.get(u["id"])
        if row is not None and row["email"] == u["email"]:
            continue
        if row is not None:
            row["valid_to"], row["is_current"] = as_of, False
        new = {"id": u["id"], "email": u["email"], "valid_from": as_of, "valid_to": None, "is_current": True}
        out.append(new)
        current[u["id"]] = new
    return sorted(out, key=lambda r: (r["id"], r["valid_from"]))

## Exercise 9: Schema drift report

**Difficulty:** Medium · about 10 min · **Practises:** [Detect schema drift](../learning/python-reference-12-data-engineering-patterns.ipynb#detect-schema-drift)

*Catch upstream schema changes before they break the load.*

Write `schema_drift(expected, records)` where expected maps field → type name (`"int"`, `"float"`, `"str"`). Return `{"added": [...], "missing": [...], "retyped": [...]}` (sorted field names): fields seen but not expected, expected but never seen with a non-None value, and fields with a non-None value of another type (an int is fine where float is expected).

**Sample calls**

| call | expected |
| --- | --- |
| `schema_drift({"id": "int", "amount": "float", "note": "str"}, [{"id": "1", "amount": 2, "coupon": "x"}])` | `{'added': ['coupon'], 'missing': ['note'], 'retyped': ['id']}` |
| `schema_drift({"id": "int"}, [{"id": 1}, {"id": 2}])` | `{'added': [], 'missing': [], 'retyped': []}` |
| `schema_drift({"id": "int", "x": "str"}, [{"id": 1, "x": None}])` | `{'added': [], 'missing': ['x'], 'retyped': []}` |

In [ ]:
def schema_drift(expected, records):
    # your code here
    pass

In [ ]:
def schema_drift(expected, records):
    seen = {}
    for r in records:
        for k, v in r.items():
            if v is not None:
                seen.setdefault(k, set()).add(type(v).__name__)
    retyped = []
    for k, t in expected.items():
        allowed = {t, "int"} if t == "float" else {t}
        if k in seen and seen[k] - allowed:
            retyped.append(k)
    return {"added": sorted(set(seen) - set(expected)), "missing": sorted(set(expected) - set(seen)), "retyped": sorted(retyped)}

## Exercise 10: Reconcile source and target counts

**Difficulty:** Easy · about 5 min · **Practises:** [Backfills and reconciliation](../learning/python-reference-12-data-engineering-patterns.ipynb#backfills-and-reconciliation)

*Post-load checks compare counts per partition and alert on mismatches.*

Write `reconcile(source, target, tolerance=0)` taking partition → count dicts and returning sorted `(partition, source_count, target_count, diff)` for partitions whose absolute difference exceeds `tolerance` (missing partitions count as 0; diff is target − source).

**Sample calls**

| call | expected |
| --- | --- |
| `reconcile({"d1": 10, "d2": 5}, {"d1": 10, "d2": 4})` | `[('d2', 5, 4, -1)]` |
| `reconcile({"d1": 3}, {"d2": 2})` | `[('d1', 3, 0, -3), ('d2', 0, 2, 2)]` |
| `reconcile({"d": 1}, {"d": 1})` | `[]` |

In [ ]:
def reconcile(source, target, tolerance=0):
    # your code here
    pass

In [ ]:
def reconcile(source, target, tolerance=0):
    out = []
    for k in sorted(source.keys() | target.keys()):
        s, t = source.get(k, 0), target.get(k, 0)
        if abs(t - s) > tolerance:
            out.append((k, s, t, t - s))
    return out

## Exercise 11: Plan a resumable backfill

**Difficulty:** Medium · about 10 min · **Practises:** [Backfills and reconciliation](../learning/python-reference-12-data-engineering-patterns.ipynb#backfills-and-reconciliation)

*Backfills over months must skip finished partitions and run in resumable chunks.*

Write `plan_backfill(start, end, done, chunk_days)` returning `(first, last)` ISO date pairs covering every date in `[start, end]` not in `done`, grouping consecutive pending dates into chunks of at most `chunk_days` days; a done date always ends a chunk.

**Sample calls**

| call | expected |
| --- | --- |
| `plan_backfill(date(2026, 11, 1), date(2026, 11, 10), set(), 4)` | `[('2026-11-01', '2026-11-04'), ('2026-11-05', '2026-11-08'), ('2026-11-09', '2026-11-10')]` |
| `plan_backfill(date(2026, 11, 1), date(2026, 11, 7), {date(2026, 11, 3)}, 7)` | `[('2026-11-01', '2026-11-02'), ('2026-11-04', '2026-11-07')]` |
| `plan_backfill(date(2026, 11, 1), date(2026, 11, 2), {date(2026, 11, 1), date(2026, 11, 2)}, 3)` | `[]` |

In [ ]:
def plan_backfill(start, end, done, chunk_days):
    # your code here
    pass

In [ ]:
from datetime import timedelta

def plan_backfill(start, end, done, chunk_days):
    chunks, current = [], []
    day = start
    while day <= end:
        if day in done:
            if current:
                chunks.append(current)
            current = []
        else:
            current.append(day)
            if len(current) == chunk_days:
                chunks.append(current)
                current = []
        day += timedelta(days=1)
    if current:
        chunks.append(current)
    return [(c[0].isoformat(), c[-1].isoformat()) for c in chunks]

## Exercise 12: Pseudonymise identifiers

**Difficulty:** Easy · about 5 min · **Practises:** [Pseudonymising personal data](../learning/python-reference-12-data-engineering-patterns.ipynb#pseudonymising-personal-data)

*Analytics needs joinable tokens, not raw emails.*

Write `pseudonymise(records, fields, key)` returning new records where each listed field (if present and not None) is replaced by the first 16 hex characters of HMAC-SHA256 of the trimmed, lower-cased value, with `key` as bytes.

**Sample calls**

| call | expected |
| --- | --- |
| `pseudonymise([{"id": 1, "email": "a@x.com"}], ["email"], b"k")` | `[{'id': 1, 'email': 'b23684d39cfdcf2a'}]` |
| `(lambda r: r[0]["email"] == r[1]["email"])(pseudonymise([{"email": " A@X.com "}, {"email": "a@x.com"}], ["email"], b"k"))` | `True` |
| `(lambda a, b: a != b)(pseudonymise([{"e": "x"}], ["e"], b"k1"), pseudonymise([{"e": "x"}], ["e"], b"k2"))` | `True` |

In [ ]:
def pseudonymise(records, fields, key):
    # your code here
    pass

In [ ]:
import hashlib
import hmac

def pseudonymise(records, fields, key):
    out = []
    for r in records:
        r = dict(r)
        for f in fields:
            if r.get(f) is not None:
                r[f] = hmac.new(key, str(r[f]).strip().lower().encode(), hashlib.sha256).hexdigest()[:16]
        out.append(r)
    return out

## Exercise 13: Batch runner with a summary

**Difficulty:** Medium · about 10 min · **Practises:** [Batches with bounded memory and a run summary](../learning/python-reference-12-data-engineering-patterns.ipynb#batches-with-bounded-memory-and-a-run-summary)

*Isolate failures per batch and report what happened.*

Write `run_batches(rows, size, handle)` splitting rows into batches of `size`, calling `handle(batch)` (returns rows written), and returning `{"batches": n, "rows_in": n, "rows_out": n, "failed": [(batch_number, error_type), ...]}` with 1-based batch numbers.

**Sample calls**

| call | expected |
| --- | --- |
| `run_batches(range(25), 10, len)` | `{'batches': 3, 'rows_in': 25, 'rows_out': 25, 'failed': []}` |
| `run_batches(range(25), 10, lambda b: len(b) if 13 not in b else 1 / 0)` | `{'batches': 3, 'rows_in': 25, 'rows_out': 15, 'failed': [(2, 'ZeroDivisionError')]}` |
| `run_batches([], 5, len)` | `{'batches': 0, 'rows_in': 0, 'rows_out': 0, 'failed': []}` |

In [ ]:
def run_batches(rows, size, handle):
    # your code here
    pass

In [ ]:
from itertools import batched

def run_batches(rows, size, handle):
    summary = {"batches": 0, "rows_in": 0, "rows_out": 0, "failed": []}
    for number, batch in enumerate(batched(rows, size), 1):
        summary["batches"] += 1
        summary["rows_in"] += len(batch)
        try:
            summary["rows_out"] += handle(list(batch))
        except Exception as error:
            summary["failed"].append((number, type(error).__name__))
    return summary

**Next:** look anything up in [Python reference 12: data engineering patterns](../learning/python-reference-12-data-engineering-patterns.ipynb), or try the [Python challenges](python-challenges-01-arrays-and-hashing.ipynb) and [Python coding problems](python-coding-problems.ipynb).